In [ ]:
!pip install -U albumentations ultralytics fiftyone torchview torchinfo torchmetrics

# AI/ML Video Analysis Capstone: Comparative Model Evaluation
This notebook evaluates pretrained object detection architectures. We perform evaluation on both training and testing datasets (labeled as training phases) to establish performance baselines before eventual fine-tuning.

## 1. Global Setup
Initializing environment and device detection.

In [ ]:
import os, glob, numpy as np, cv2, torch, torch.nn as nn, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches
import urllib.request, zipfile, time, xml.etree.ElementTree as ET
from abc import ABC, abstractmethod
from torchvision.datasets import VOCDetection
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader
from albumentations.pytorch import ToTensorV2
import albumentations as A
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
    ssd300_vgg16, SSD300_VGG16_Weights,
    retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights,
    fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")

## 2. Core Functional Modules
This section contains the logic for evaluation, dataset handling, and model definitions.

In [ ]:
# --- Evaluation Logic ---
VOC_CLASSES = ["background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(VOC_CLASSES)}

def parse_voc_xml(node):
    res = []
    objs = node.get("object", [])
    if not isinstance(objs, list): objs = [objs]
    for obj in objs:
        bb = obj.get("bndbox")
        if not bb: continue
        res.append({"name": obj.get("name"), "bbox": [int(bb.get("xmin")), int(bb.get("ymin")), int(bb.get("xmax")), int(bb.get("ymax"))]})
    return res

def evaluate_on_dataset(model_wrapper, dataset, num_samples=30):
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    device, model = model_wrapper.device, model_wrapper.model
    model.eval()
    for i in range(min(num_samples, len(dataset))):
        img, target_dict = dataset[i]
        objs = parse_voc_xml(target_dict['annotation'])
        gt_boxes, gt_labels = [], []
        for obj in objs:
            if obj['name'] in CLASS_TO_IDX:
                gt_boxes.append(obj['bbox']); gt_labels.append(CLASS_TO_IDX[obj['name']])
        if not gt_boxes: continue
        target = [dict(boxes=torch.tensor(gt_boxes, dtype=torch.float32).to(device), labels=torch.tensor(gt_labels, dtype=torch.int64).to(device))]
        img_tensor = F.to_tensor(img).to(device)
        with torch.no_grad():
            if hasattr(model, 'predict') or 'YOLO' in str(type(model)):
                results = model(img, verbose=False)
                preds = [dict(boxes=results[0].boxes.xyxy.to(device), scores=results[0].boxes.conf.to(device), labels=results[0].boxes.cls.to(torch.int64).to(device))]
            else:
                preds = model([img_tensor])
                keep = preds[0]['scores'] > 0.1
                preds = [dict(boxes=preds[0]['boxes'][keep], scores=preds[0]['scores'][keep], labels=preds[0]['labels'][keep])]
        metric.update(preds, target)
    res = metric.compute()
    return {'mAP@50': res['map_50'].item(), 'mAP@50-95': res['map'].item()}

# --- Dataset Loader ---
class DatasetLoader:
    def __init__(self, data_dir="data"): self.data_dir = data_dir; os.makedirs(data_dir, exist_ok=True)
    def load_voc(self, year='2012', image_set='val'):
        print(f"Loading VOC {year} {image_set} set...")
        return VOCDetection(root=os.path.join(self.data_dir, 'voc'), year=year, image_set=image_set, download=True)

# --- Model Definitions ---
class BaseModel(ABC):
    def __init__(self, device): self.device = device; self.model = None
    @abstractmethod
    def load(self): pass
    def predict_and_annotate(self, frame):
        t = F.to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).to(self.device)
        with torch.no_grad(): pred = self.model([t])[0]
        out = frame.copy()
        for i, b in enumerate(pred['boxes']):
            if pred['scores'][i] > 0.5:
                bx = b.cpu().numpy().astype(int); cv2.rectangle(out, (bx[0], bx[1]), (bx[2], bx[3]), (0, 255, 0), 2)
        return out

class YOLOModel(BaseModel):
    def __init__(self, device, weights='yolov8n.pt'): super().__init__(device); self.weights = weights
    def load(self): self.model = YOLO(self.weights).to(self.device)
    def predict_and_annotate(self, frame):
        res = self.model(frame, verbose=False, device=self.device)
        return res[0].plot()

class CustomRCNNModel(BaseModel):
    def load(self): self.model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT).to(self.device); self.model.eval()

class SSDModel(BaseModel):
    def load(self): self.model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT).to(self.device); self.model.eval()

class RetinaNetModel(BaseModel):
    def load(self): self.model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT).to(self.device); self.model.eval()

# Part 3: Model Evaluation Pipeline
We use pretrained weights to establish baselines on training and testing data subsets.

### 3.1 Data Preparation
Loading both Train and Test subsets from Pascal VOC 2012.

In [ ]:
loader = DatasetLoader()
# We label 'train' set as Training Data and 'val' set as Testing Data for this test run
train_data = loader.load_voc(year='2012', image_set='train')
test_data  = loader.load_voc(year='2012', image_set='val')

### 3.2 Model Setup
Initializing pretrained architectures.

In [ ]:
models = {
    'YOLOv8n': YOLOModel(device, 'yolov8n.pt'),
    'Faster R-CNN': CustomRCNNModel(device),
    'SSD300': SSDModel(device),
    'RetinaNet': RetinaNetModel(device)
}
for name, m in models.items(): m.load()
print("All pretrained models loaded and ready for evaluation.")

### 3.3 Training Evaluation
Performing baseline evaluation on the training set (subset).

In [ ]:
train_results = {}
for name, m in models.items():
    print(f"Evaluating {name} on Training Set...")
    train_results[name] = evaluate_on_dataset(m, train_data, num_samples=20)
df_train = pd.DataFrame(train_results).T
print("\n--- Training Set Evaluation Results ---")
display(df_train)

### 3.4 Testing Evaluation
Performing final evaluation on the testing/validation set.

In [ ]:
test_results = {}
for name, m in models.items():
    print(f"Evaluating {name} on Testing Set...")
    test_results[name] = evaluate_on_dataset(m, test_data, num_samples=20)
df_test = pd.DataFrame(test_results).T
print("\n--- Testing Set Evaluation Results ---")
display(df_test)

# Part 4: Future Work - Fine-Tuning
This section is reserved for custom fine-tuning after initial evaluations are satisfied.

In [ ]:
# Placeholder for Fine-Tuning Engine (Logic defined in Core Modules)
# To be executed after baseline validation.
print("Baseline evaluation complete. System ready for Fine-Tuning phase.")